# Employee Attrition Analysis — IBM HR Analytics
## Exploratory Data Analysis (EDA)

**Question:** What actually predicts attrition here — and which "obvious" factors turn out not to matter?

Dataset: [IBM HR Analytics Attrition, Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.shape

(1470, 35)

## 1. First look

In [2]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

## 2. Data quality checks — judgment calls

In [4]:
# Check for near-constant columns — these carry no signal and should be dropped
for col in df.columns:
    if df[col].nunique() <= 2:
        print(f"{col}: {df[col].unique()}")

Attrition: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
EmployeeCount: [1]
Gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Over18: <StringArray>
['Y']
Length: 1, dtype: str
OverTime: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
PerformanceRating: [3 4]
StandardHours: [80]


**Judgment call 1 — near-constant columns:** `EmployeeCount` (always 1), `Over18` (always 'Y'), and `StandardHours` (always 80) carry no information — every row has the same value. Dropping these three; keeping other binary columns since they contain real variation (e.g. `Attrition`, `OverTime`).

Also worth noting: `PerformanceRating` only takes values 3 or 4 in this dataset — a real limitation of the data, not a cleaning issue, but it means this feature has less discriminating power than its 1–4 scale suggests.

In [5]:
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours"])
df.shape

(1470, 32)

In [6]:
print(f"EmployeeNumber unique values: {df['EmployeeNumber'].nunique()} out of {len(df)} rows")

EmployeeNumber unique values: 1470 out of 1470 rows


**Judgment call 2 — identifier column:** `EmployeeNumber` is unique for every row (1,470/1,470) — a pure ID with no predictive signal. Dropping it before modeling, same logic as excluding `Row ID` in the Superstore project.

In [7]:
df = df.drop(columns=["EmployeeNumber"])
df.shape

(1470, 31)

## 3. Target variable — class imbalance check

In [8]:
df["Attrition"].value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [9]:
df["Attrition"].value_counts(normalize=True)

Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

**Judgment call 3 — class imbalance:** Attrition is imbalanced (83.9% No, 16.1% Yes — roughly 5:1). This has two consequences for later:
1. Accuracy alone would be misleading — a model predicting "No" for every employee would score 84% accuracy while catching zero at-risk employees
2. Precision/recall (or a metric like F1 or ROC-AUC) will be used instead to evaluate any classifier, since they account for the minority class specifically

No resampling (SMOTE, undersampling) is applied at this stage — the imbalance is moderate enough that class-weighting in the model, and honest evaluation via precision/recall, should be sufficient without introducing synthetic data.

## 4. Does overtime predict attrition?

Starting with the most commonly assumed driver of attrition.

In [10]:
pd.crosstab(df["OverTime"], df["Attrition"], normalize="index")

Attrition,No,Yes
OverTime,,
No,0.895636,0.104364
Yes,0.694712,0.305288


Employees who work overtime have a 30.5% attrition rate — almost 3x higher than those who don't (10.4%). This is a strong candidate driver. But before trusting it as an actionable finding, the natural follow-up question is: **is this overtime effect consistent across every department, or is it concentrated in one place?**

In [11]:
overtime_by_dept = df.groupby(["Department", "OverTime"])["Attrition"].apply(
    lambda x: (x == "Yes").mean()
).unstack()
overtime_by_dept

OverTime,No,Yes
Department,,
Human Resources,0.152174,0.294118
Research & Development,0.085507,0.273063
Sales,0.138365,0.375000


The overtime effect holds across every department — not concentrated in one place:

| Department | No overtime | Overtime |
|---|---|---|
| Sales | 13.8% | 37.5% |
| Human Resources | 15.2% | 29.4% |
| R&D | 8.6% | 27.3% |

Sales has both the highest overall attrition and the widest overtime gap (13.8% → 37.5%). R&D has the lowest baseline attrition but still nearly triples when overtime is involved. This confirms overtime is a genuine, company-wide driver — not a single-department artifact — which makes it a solid candidate for the model and for the final recommendation.

In [12]:
df.groupby("JobSatisfaction")["Attrition"].apply(lambda x: (x == "Yes").mean())

JobSatisfaction
1    0.228374
2    0.164286
3    0.165158
4    0.113290
Name: Attrition, dtype: float64

In [13]:
df.groupby("Attrition")["MonthlyIncome"].median()

Attrition
No     5204.0
Yes    3202.0
Name: MonthlyIncome, dtype: float64

## 5. Job satisfaction and income — real, but weaker or more tangled than expected

**Job satisfaction:** shows a real gradient but not a clean one. Level 1 (lowest) has 22.8% attrition — clearly worse — but levels 2 and 3 are nearly identical (16.4% vs 16.5%), and level 4 drops to 11.3%. This is a real factor but a much weaker signal than overtime, and not linear.

**Monthly income:** employees who left had a lower median income ($3,202 vs $5,204, about 38% less). This looks strong, but income is entangled with job level and tenure — junior employees naturally earn less AND tend to leave more for unrelated reasons. This needs to be checked against job level before treating income as an independent driver, rather than a proxy for seniority.

In [14]:
df.groupby(["JobLevel", "Attrition"])["MonthlyIncome"].median().unstack()

Attrition,No,Yes
JobLevel,,
1,2719.0,2437.0
2,5334.5,5346.0
3,9982.5,9887.0
4,16307.0,13194.0
5,19199.5,19545.0


Once controlled for job level, the income gap mostly disappears. At Levels 1, 2, 3, and 5, income is nearly identical between employees who stayed and left. Only Level 4 shows a meaningful gap ($16,307 vs $13,194).

**This confirms the suspicion:** the raw income gap wasn't really about pay — it was a seniority proxy. Junior employees (who are naturally paid less) leave more often for reasons unrelated to their specific salary. **This is exactly the kind of "obvious factor that turns out to matter less than expected" the project set out to find** — income looked like a strong standalone driver but is largely explained by job level instead.

## 6. Classifier — predicting attrition

Using logistic regression with class weighting (to account for the 5:1 imbalance) rather than plain accuracy. Features chosen based on what the EDA actually supported: OverTime, JobSatisfaction, JobLevel, plus a few standard demographic/tenure variables for context.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

model_df = df.copy()

model_df["Attrition"] = (model_df["Attrition"] == "Yes").astype(int)
model_df["OverTime"] = (model_df["OverTime"] == "Yes").astype(int)

features = ["OverTime", "JobSatisfaction", "JobLevel", "Age", "TotalWorkingYears",
            "YearsAtCompany", "MonthlyIncome", "DistanceFromHome", "WorkLifeBalance"]

X = model_df[features]
y = model_df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((1102, 9), (368, 9))

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
y_prob = clf.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Stayed", "Left"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

              precision    recall  f1-score   support

      Stayed       0.92      0.72      0.81       309
        Left       0.32      0.68      0.43        59

    accuracy                           0.72       368
   macro avg       0.62      0.70      0.62       368
weighted avg       0.83      0.72      0.75       368

ROC-AUC: 0.750


In [18]:
coef_df = pd.DataFrame({
    "feature": features,
    "coefficient": clf.coef_[0]
}).sort_values("coefficient", ascending=False)
coef_df

,feature,coefficient
0,OverTime,0.594988
2,JobLevel,0.223551
7,DistanceFromHome,0.212816
4,TotalWorkingYears,-0.168433
8,WorkLifeBalance,-0.210215
5,YearsAtCompany,-0.218457
1,JobSatisfaction,-0.294258
6,MonthlyIncome,-0.326726
3,Age,-0.347250


## Feature importance

| Feature | Coefficient | Direction |
|---|---|---|
| OverTime | +0.59 | Strongest predictor — confirms EDA |
| JobLevel | +0.22 | Higher level → slightly more likely to leave (counter-intuitive) |
| DistanceFromHome | +0.21 | New finding — not explored in EDA, longer commute → more attrition |
| Age | -0.35 | Strongest negative predictor — younger employees leave more |
| MonthlyIncome | -0.33 | Higher income → less attrition (independent of the job-level confound found earlier) |
| JobSatisfaction | -0.29 | Confirms EDA — matters, but not dominant |
| YearsAtCompany | -0.22 | More tenure → less attrition |
| WorkLifeBalance | -0.21 | Better balance → less attrition |
| TotalWorkingYears | -0.17 | More career experience → less attrition |

OverTime remains the standout single factor even after controlling for everything else, confirming the EDA finding rather than explaining it away. Age emerges as the strongest predictor overall — younger employees are meaningfully more attrition-prone, independent of tenure or income. DistanceFromHome is a genuinely new finding not surfaced during EDA — worth a follow-up look.

## So what

Three things HR could act on this quarter:
1. **Overtime policy** — the single strongest, most consistent driver across every department (13.8–37.5% attrition swing). Review overtime load in Sales specifically, where both baseline attrition and the overtime gap are worst.
2. **Early-career retention** — age is the strongest independent predictor; younger/newer employees need targeted retention efforts (mentorship, career-pathing) rather than blanket policies aimed at the whole workforce.
3. **Commute distance** — a new finding worth investigating further (remote/hybrid flexibility for employees with long commutes) rather than acting on immediately, since it wasn't part of the original hypothesis and deserves its own follow-up.

**What didn't hold up:** the raw income gap between leavers and stayers was largely explained by job level, not income itself — a caution against treating a bivariate correlation as an independent driver without checking for confounds.